In [ ]:
!pip -q install transformers datasets evaluate senetencepiece sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.8 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset

raw = load_dataset("dim/grammarly_coedit")
print(raw)

if "validation" not in raw:
    ds = raw["train"].train_test_split(test_size=0.1, seed=42)
    train_ds = ds["train"]
    val_ds   = ds["test"]
else:
    train_ds = raw["train"]
    val_ds   = raw["validation"]

print(f"Train rows: {len(train_ds)},  Val rows: {len(val_ds)}")

DatasetDict({
    train: Dataset({
        features: ['_id', 'task', 'src', 'tgt'],
        num_rows: 82466
    })
})
Train rows: 74219,  Val rows: 8247


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "t5-small"
PREFIX = "grammar: "

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(batch):
    inputs = [PREFIX + s for s in batch["src"]]    # 'src' column
    m = tok(inputs, truncation=True, max_length=128)
    with tok.as_target_tokenizer():
        labels = tok(batch["tgt"], truncation=True, max_length=128)
    m["labels"] = labels["input_ids"]
    return m

tok_train = train_ds.map(preprocess, batched=True,
                         remove_columns=train_ds.column_names)
tok_val   = val_ds.map(preprocess,   batched=True,
                       remove_columns=val_ds.column_names)

print("Train columns:", tok_train.column_names)
print("Val columns:  ", tok_val.column_names)

Map:   0%|          | 0/74219 [00:00<?, ? examples/s]

Map:   0%|          | 0/8247 [00:00<?, ? examples/s]

Columns in tokenised train: ['input_ids', 'attention_mask', 'labels']
Columns in tokenised val:   ['input_ids', 'attention_mask', 'labels']


In [ ]:
from transformers import DataCollatorForSeq2Seq
import evaluate, numpy as np, torch

collator = DataCollatorForSeq2Seq(tok, model=None)
gleu = evaluate.load("google_bleu")

PAD, V = tok.pad_token_id, len(tok) - 1
def _fix():
  if isinstance(x,(list,tuple)): return [_fix() for i in x]
  if isinstance(x,(np.ndarray,torch.Tensor)): return [_fix() for i in x.tolist()]
  i = int(x); return i if 0 <= i <= V else PAD

def compute_metrics(pred):
    preds, labels = pred
    preds  = _fix(preds)
    labels = _fix(np.where(labels!=-100, labels, PAD))
    sys  = tok.batch_decode(preds, skip_special_tokens=True)
    refs = tok.batch_decode(labels, skip_special_tokens=True)
    return {"gleu": gleu.compute(predictions=sys, references=[[r] for r in refs])["google_bleu"]}

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

args = Seq2SeqTrainingArguments(
    output_dir             = "./t5_grammarly_coedit",
    num_train_epochs       = 4,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    gradient_accumulation_steps = 2,
    learning_rate          = 3e-4,
    eval_strategy          = "steps",
    eval_steps             = 500,
    save_strategy          = "steps",
    save_steps             = 500,
    save_total_limit       = 2,
    load_best_model_at_end = True,
    metric_for_best_model  = "eval_gleu",
    greater_is_better      = True,
    generation_max_length  = 128,
    predict_with_generate  = True,
    fp16                   = torch.cuda.is_available(),
    logging_steps          = 200,
    report_to              = "none",
)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
from transformers import Seq2SeqTrainer, EarlyStoppingCallback, TrainerCallback

class LogCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        if logs and state.is_world_process_zero:
            parts = []
            for k, v in logs.items():
                if "gleu" in k:
                    parts.append(f"{k}:{v:6.2f}")
                elif "learning_rate" in k:
                    parts.append(f"{k}:{v:8.6f}")
                elif isinstance(v, (int, float)):
                    parts.append(f"{k}:{v:6.3f}")
            print(f"step{state.global_step: > 6} | " + " | ".join(parts))

trainer = Seq2SeqTrainer(
    model            = model,
    args             = args,
    train_dataset    = tok_train,
    eval_dataset     = tok_val,
    tokenizer        = tok,
    data_collator    = collator,
    compute_metrics  = compute_metrics,
    callbacks        = [
        EarlyStoppingCallback(early_stopping_patience=2),
        LogCallback()
    ],
)

print("Success")

<ipython-input-13-572f2815bf31>:13: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


✓ Trainer ready — call `trainer.train()`


In [ ]:
trainer.train()
trainer.save_model()
tok.save_pretrained("./t5_grammarly_coedit")
print("Model saved to ./t5_grammarly_coedit")
print("Validation GLEUL", trainer.evaluate()["eval_gleu"])
print("Validation GLEU:", trainer.evaluate()["eval_gleu"])


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss,Validation Loss,Gleu
500,1.103100,0.959668,0.578471
1000,1.031800,0.932052,0.585025
1500,1.008200,0.918795,0.592905
2000,0.987100,0.898558,0.596797
2500,0.961400,0.885002,0.599330
3000,0.933300,0.876197,0.599841
3500,0.948200,0.872396,0.604153
4000,0.988800,0.865905,0.605354
4500,0.944100,0.865959,0.606975
5000,0.865900,0.858663,0.608573


step    200 | loss: 1.217 | grad_norm: 0.865 | learning_rate:0.000297 | epoch: 0.043
step    400 | loss: 1.103 | grad_norm: 1.627 | learning_rate:0.000294 | epoch: 0.086
step    500 | eval_loss: 0.960 | eval_gleu:  0.58 | eval_runtime:633.229 | eval_samples_per_second:13.024 | eval_steps_per_second: 1.628 | epoch: 0.108
step    600 | loss: 1.075 | grad_norm: 0.993 | learning_rate:0.000290 | epoch: 0.129
step    800 | loss: 1.060 | grad_norm: 1.230 | learning_rate:0.000287 | epoch: 0.172
step   1000 | loss: 1.032 | grad_norm: 1.377 | learning_rate:0.000284 | epoch: 0.216
step   1000 | eval_loss: 0.932 | eval_gleu:  0.59 | eval_runtime:645.548 | eval_samples_per_second:12.775 | eval_steps_per_second: 1.597 | epoch: 0.216
step   1200 | loss: 1.008 | grad_norm: 1.323 | learning_rate:0.000281 | epoch: 0.259
step   1400 | loss: 1.008 | grad_norm: 1.447 | learning_rate:0.000277 | epoch: 0.302
step   1500 | eval_loss: 0.919 | eval_gleu:  0.59 | eval_runtime:635.282 | eval_samples_per_second:12

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


step   7500 | train_runtime:10540.351 | train_samples_per_second:28.166 | train_steps_per_second: 1.760 | total_flos:1710439113424896.000 | train_loss: 0.953 | epoch: 1.617
✔ Model saved to ./t5_grammarly_coedit


step   7500 | eval_loss: 0.848 | eval_gleu:  0.61 | eval_runtime:628.341 | eval_samples_per_second:13.125 | eval_steps_per_second: 1.641 | epoch: 1.617
Validation GLEU: 0.6119451053824602


In [ ]:
def correction(text: str):
    inp = tok(PREFIX + text, return_tensors="pt").to(model.device)
    out = model.generate(**inp, num_beams=5, max_length=128)
    return tok.decode(out[0], skip_special_tokens=True)


print(correction("She go to school yesterday."))
print(correction("The smal dog juped over the fence yesterday."))


SRC: She go to school yesterday.
→ She went to school yesterday.
SRC: Teh smal dog juped over teh fence yestarday.
→ Teh smal dog juped over teh fence yestarday.
